# Post-Processing Demo

Demonstrates how overlap-removal post-processors affect Matcher predictions.

| # | Post-Processor | Key Parameter |
|---|----------------|---------------|
| 1 | BoxNMS | `iou_threshold=0.1` |
| 2 | MaskIoMNMS | `iom_threshold=0.5` |
| 3 | BoxIoMNMS | `iom_threshold=0.5` |

In [ ]:
# Copyright (C) 2026 Intel Corporation
# SPDX-License-Identifier: Apache-2.0

from __future__ import annotations

import colorsys
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from instantlearn.components.postprocessing import BoxIoMNMS, BoxNMS, MaskIoMNMS
from instantlearn.data import Sample
from instantlearn.data.utils.image import read_image
from instantlearn.models import Matcher
from instantlearn.visualizer import render_predictions

## Helpers

In [ ]:
EXAMPLES_DIR = Path("assets/coco")


def select_device() -> str:
    """Auto-select the best available device."""
    if hasattr(torch, "xpu") and torch.xpu.is_available():
        return "xpu"
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"


def instance_cmap(n: int) -> dict[int, list[int]]:
    """Generate a unique colour for each of *n* masks."""
    cmap: dict[int, list[int]] = {}
    for i in range(n):
        rgb = colorsys.hsv_to_rgb(i / max(n, 1), 0.9, 0.95)
        cmap[i] = [int(c * 255) for c in rgb]
    return cmap


def show(
    images: list[torch.Tensor],
    predictions: list[dict[str, torch.Tensor]],
    title: str,
) -> None:
    """Plot images with overlaid mask predictions and a count/score summary."""
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(7 * n, 7))
    if n == 1:
        axes = [axes]
    for ax, img, pred in zip(axes, images, predictions, strict=False):
        pred_copy = {**pred}
        n_masks = pred_copy["pred_masks"].shape[0]
        scores = pred_copy["pred_scores"]
        pred_copy["pred_labels"] = torch.arange(n_masks)
        cmap = instance_cmap(n_masks)
        vis = render_predictions(img.permute(1, 2, 0).numpy(), pred_copy, cmap)
        ax.imshow(vis)
        ax.set_title(f"{n_masks} masks  [{scores.min():.2f}, {scores.max():.2f}]", fontsize=12)
        ax.axis("off")
    fig.suptitle(title, fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.show()

## Load reference & targets

In [ ]:
device = select_device()

ref_sample = Sample(
    image_path=str(EXAMPLES_DIR / "000000286874.jpg"),
    mask_paths=str(EXAMPLES_DIR / "000000286874_mask.png"),
)

target_paths = [
    str(EXAMPLES_DIR / "000000390341.jpg"),
    str(EXAMPLES_DIR / "000000173279.jpg"),
    str(EXAMPLES_DIR / "000000267704.jpg"),
]
target_images = [read_image(p) for p in target_paths]

## Original images

In [ ]:
model = Matcher(device=device, postprocessor=None)
model.fit(ref_sample)
raw = model.predict(target_paths)

show(target_images, raw, "Raw predictions (no post-processing)")

## Raw predictions (no post-processing)

In [ ]:
model = Matcher(device=device, postprocessor=None)
model.fit(ref_sample)
raw = model.predict(target_paths)

show(target_images, raw, "Raw predictions (no post-processing)")

## 1. BoxNMS (IoU threshold = 0.1)

Attach a `BoxNMS` post-processor to the model. Subsequent `predict()` calls
automatically apply it — boxes with IoU > threshold are suppressed.


In [ ]:
model.postprocessor = BoxNMS(iou_threshold=0.1)
results = model.predict(target_paths)

show(target_images, results, "BoxNMS (iou_threshold=0.1)")

## 2. MaskIoMNMS (IoM threshold = 0.5)

Replace the post-processor with `MaskIoMNMS`. The *smaller* mask is
suppressed when the overlap (relative to its area) exceeds the threshold.


In [ ]:
model.postprocessor = MaskIoMNMS(iom_threshold=0.5)
results = model.predict(target_paths)

show(target_images, results, "MaskIoMNMS (iom_threshold=0.5)")

## 3. BoxIoMNMS (IoM threshold = 0.5)

Same idea as `MaskIoMNMS` but computed on bounding boxes, which is
faster for large mask counts.


In [ ]:
model.postprocessor = BoxIoMNMS(iom_threshold=0.5)
results = model.predict(target_paths)

show(target_images, results, "BoxIoMNMS (iom_threshold=0.5)")